In [ ]:
from hashlib import shake_256
from pathlib import Path
from typing import Sequence
import random
import time

import chipwhisperer as cw


# Current firmware uses the official LESS toy parameters.
Q = 127
N = 16
K = 8

ROW_LENGTH = N - K

SEED_LENGTH_BYTES = 16

FQ_ELEM_BYTES = 1
POSITION_T_BYTES = 2

VALUE_BYTES = (
    K * ROW_LENGTH * FQ_ELEM_BYTES
)

POSITION_BYTES = (
    ROW_LENGTH * POSITION_T_BYTES
)

INPUT_BYTES = SEED_LENGTH_BYTES

OUTPUT_BYTES = (
    VALUE_BYTES
    + POSITION_BYTES
)

SUBROUTINE_GENERATOR_SAMPLE = 0x03

BAUDRATE = 230400

FIRMWARE_DIRECTORY = Path.cwd()

HEX_BINARY = (
    FIRMWARE_DIRECTORY
    / "less_generator_sample-PINATA.hex"
)

print("Seed bytes    :", INPUT_BYTES)
print("Value bytes   :", VALUE_BYTES)
print("Position bytes:", POSITION_BYTES)
print("Output bytes  :", OUTPUT_BYTES)
print("HEX file      :", HEX_BINARY)

In [ ]:
class ShakeReader:
    """Sequential reader over Python's SHAKE256 output stream."""

    def __init__(
        self,
        seed: bytes,
    ) -> None:
        self._shake = shake_256(seed)
        self._offset = 0

    def read(
        self,
        length: int,
    ) -> bytes:
        end = self._offset + length

        data = self._shake.digest(end)[
            self._offset:end
        ]

        self._offset = end

        return data


def serialize_generator_sample_seed(
    seed: Sequence[int] | bytes | bytearray,
) -> bytes:
    """Validate and serialize one LESS generator seed."""

    try:
        seed_bytes = bytes(seed)
    except (TypeError, ValueError) as exc:
        raise TypeError(
            "Seed must be bytes or a sequence of byte values"
        ) from exc

    if len(seed_bytes) != SEED_LENGTH_BYTES:
        raise ValueError(
            f"Expected {SEED_LENGTH_BYTES} seed bytes, "
            f"got {len(seed_bytes)}"
        )

    return seed_bytes


def generator_sample_python(
    seed: Sequence[int] | bytes | bytearray,
) -> tuple[list[int], list[int]]:
    """Independent reproduction of LESS generator_sample."""

    seed_bytes = serialize_generator_sample_seed(
        seed
    )

    reader = ShakeReader(
        seed_bytes
    )

    values: list[int] = []

    for _ in range(K):
        row: list[int] = []

        while len(row) < ROW_LENGTH:
            word = int.from_bytes(
                reader.read(8),
                byteorder="little",
            )

            # Extract nine complete 7-bit candidates from each
            # 64-bit word. Candidate 127 is rejected.
            for _ in range(64 // 7):
                candidate = word & 0x7F

                if candidate <= Q - 1:
                    row.append(candidate)

                    if len(row) == ROW_LENGTH:
                        break

                word >>= 7

        values.extend(row)

    column_positions = list(
        range(K, N)
    )

    return (
        values,
        column_positions,
    )


def deserialize_generator_sample_result(
    payload: bytes | bytearray,
) -> tuple[list[int], list[int]]:
    """Deserialize the 80-byte firmware response."""

    payload_bytes = bytes(
        payload
    )

    if len(payload_bytes) != OUTPUT_BYTES:
        raise ValueError(
            f"Expected {OUTPUT_BYTES} result bytes, "
            f"got {len(payload_bytes)}"
        )

    values = list(
        payload_bytes[:VALUE_BYTES]
    )

    position_data = payload_bytes[
        VALUE_BYTES:
    ]

    positions = [
        int.from_bytes(
            position_data[
                POSITION_T_BYTES * index:
                POSITION_T_BYTES * (index + 1)
            ],
            byteorder="little",
            signed=False,
        )
        for index in range(ROW_LENGTH)
    ]

    return (
        values,
        positions,
    )

In [ ]:
def connect_and_flash():
    """Connect, configure the clock, flash, and create the target."""

    if not HEX_BINARY.is_file():
        raise FileNotFoundError(
            f"Firmware image not found: {HEX_BINARY}"
        )

    scope = cw.scope()
    scope.default_setup()

    scope.io.clkout = 15e6

    print(
        "Clock output:",
        scope.io.clkout,
    )

    time.sleep(0.5)

    cw.program_target(
        scope,
        cw.programmers.STM32FProgrammer,
        str(HEX_BINARY),
        baud=BAUDRATE,
    )

    target = cw.target(
        scope,
        cw.targets.SimpleSerial2,
    )

    target.baud = BAUDRATE
    target.flush()

    return (
        scope,
        target,
    )


def upload_generator_sample_seed(
    target,
    seed: Sequence[int] | bytes | bytearray,
    verify_readback: bool = True,
) -> None:
    """Upload the complete generator_sample seed."""

    payload = serialize_generator_sample_seed(
        seed
    )

    subcommand = (
        0x80
        if verify_readback
        else 0x00
    )

    target.send_cmd(
        "s",
        subcommand,
        bytearray(payload),
    )

    if verify_readback:
        readback = target.simpleserial_read(
            "r",
            INPUT_BYTES,
            timeout=1000,
            ack=True,
        )

        if readback is None:
            raise TimeoutError(
                "No generator_sample seed readback received"
            )

        if bytes(readback) != payload:
            raise RuntimeError(
                "generator_sample seed readback mismatch"
            )

    else:
        acknowledgement = (
            target.simpleserial_wait_ack(
                timeout=1000
            )
        )

        if acknowledgement is None:
            raise TimeoutError(
                "No acknowledgement for generator_sample seed"
            )


def generator_sample_board(
    target,
    seed: Sequence[int] | bytes | bytearray,
) -> tuple[list[int], list[int]]:
    """Execute generator_sample on the Pinata."""

    upload_generator_sample_seed(
        target,
        seed,
        verify_readback=True,
    )

    target.send_cmd(
        "b",
        SUBROUTINE_GENERATOR_SAMPLE,
        bytearray(),
    )

    response = target.simpleserial_read(
        "r",
        OUTPUT_BYTES,
        timeout=2000,
        ack=True,
    )

    if response is None:
        raise TimeoutError(
            "No result received from generator_sample"
        )

    if len(response) != OUTPUT_BYTES:
        raise RuntimeError(
            f"Expected {OUTPUT_BYTES} result bytes, "
            f"received {len(response)}"
        )

    return deserialize_generator_sample_result(
        response
    )

In [ ]:
scope = None
target = None

try:
    scope, target = connect_and_flash()

    deterministic_seed = bytes(
        range(SEED_LENGTH_BYTES)
    )

    expected = generator_sample_python(
        deterministic_seed
    )

    actual = generator_sample_board(
        target,
        deterministic_seed,
    )

    print(
        "Expected first row:",
        expected[0][:ROW_LENGTH],
    )

    print(
        "Board first row   :",
        actual[0][:ROW_LENGTH],
    )

    print(
        "Expected positions:",
        expected[1],
    )

    print(
        "Board positions   :",
        actual[1],
    )

    if actual != expected:
        raise AssertionError(
            "Pinata result does not match the Python result"
        )

    number_of_random_tests = 100

    for test_index in range(
        number_of_random_tests
    ):
        seed = bytes(
            random.randrange(256)
            for _ in range(SEED_LENGTH_BYTES)
        )

        expected = generator_sample_python(
            seed
        )

        actual = generator_sample_board(
            target,
            seed,
        )

        if actual != expected:
            raise AssertionError(
                f"Mismatch in random test {test_index}"
            )

    print(
        f"OK: deterministic test and "
        f"{number_of_random_tests} random tests passed."
    )

finally:
    if target is not None:
        target.dis()

    if scope is not None:
        scope.dis()